# Package the raw LLM predictions

Measures the cached prediction arrays, bundles each backbone into one
compressed `.npz`, and reports which hosting route the sizes allow.

| route | per-file limit | notes |
|---|---|---|
| plain git | **100 MB hard**, warns over 50 MB | stays in history forever |
| GitHub release asset | **2 GB** | not counted against repo size, no DOI |
| Git LFS | 2 GB | 1 GB free storage and bandwidth, then paid |
| Zenodo | 50 GB | citable DOI |


In [ ]:
SOURCE_ROOT = ''      # <- folder holding the original runs
WRITE_NPZ   = True    # False = measure only, write nothing

import os, sys, glob
import numpy as np

if 'google.colab' in sys.modules or os.path.isdir('/content'):
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
    RELEASE = '/content/drive/MyDrive/sdg-llm-graph'
else:
    RELEASE = os.path.abspath('..')

MODELS = ['gemma4-26b', 'mistral-24b', 'qwen3-32b', 'mixtral-8x7b']
OUTDIR = os.path.join(RELEASE, 'artifacts', 'predictions')
READY  = False

if not SOURCE_ROOT:
    base = '/content/drive/MyDrive'
    cands = [d for d in sorted(os.listdir(base))
             if os.path.isdir(os.path.join(base, d, 'aurora_sdg_graph_full'))]
    print('SOURCE_ROOT is not set.')
    print('Folders that contain aurora_sdg_graph_full/:')
    for c in cands:
        print(f'    {base}/{c}')
    print('\nPut one of those paths in SOURCE_ROOT above, then Run all again.')
    print('(Later cells will skip themselves until then — this is not an error.)')
elif not os.path.isdir(os.path.join(SOURCE_ROOT, 'aurora_sdg_graph_full')):
    print(f'No aurora_sdg_graph_full/ under {SOURCE_ROOT} — check the path.')
else:
    READY = True
    AUR = os.path.join(SOURCE_ROOT, 'aurora_sdg_graph_full')
    print('source :', AUR)
    print('output :', OUTDIR)


## 1. Inventory

In [ ]:
def mb(n): return n / (1024 ** 2)
inventory, total_raw = {}, 0

if not READY:
    print('skipped — SOURCE_ROOT not set')
else:
    for m in MODELS:
        d = os.path.join(AUR, f'results_v5_multi_{m}_n10000')
        if not os.path.isdir(d):
            print(f'{m:14s} results dir MISSING')
            continue
        files = sorted(glob.glob(os.path.join(d, '*.npy')))
        inventory[m] = files
        sub = sum(os.path.getsize(f) for f in files)
        total_raw += sub
        print(f'{m:14s} {len(files)} .npy   {mb(sub):8.1f} MB')
        for f in files:
            a = np.load(f, mmap_mode='r')
            print(f'                 {os.path.basename(f):58s} '
                  f'{str(a.shape):16s} {str(a.dtype):8s} {mb(os.path.getsize(f)):7.1f} MB')
    print(f'\nTOTAL RAW: {mb(total_raw):.1f} MB across '
          f'{sum(len(v) for v in inventory.values())} files')


## 2. Bundle each backbone into one compressed `.npz`

In [ ]:
total_npz = 0

if not READY or not inventory:
    print('skipped — nothing to package')
else:
    os.makedirs(OUTDIR, exist_ok=True)
    for m, files in inventory.items():
        out = os.path.join(OUTDIR, f'raw_llm_predictions_{m}_n10000.npz')
        if WRITE_NPZ:
            arrays = {os.path.splitext(os.path.basename(f))[0]: np.load(f)
                      for f in files}
            np.savez_compressed(out, **arrays)
        if os.path.exists(out):
            sz = os.path.getsize(out)
            raw = sum(os.path.getsize(f) for f in files)
            total_npz += sz
            print(f'{m:14s} {mb(raw):7.1f} MB -> {mb(sz):7.1f} MB  '
                  f'({100 * sz / raw:.0f}% of raw)')
    print(f'\nTOTAL PACKAGED: {mb(total_npz):.1f} MB')


## 3. Which hosting route fits

In [ ]:
largest, sizes = 0, []
if os.path.isdir(OUTDIR):
    sizes = [(f, os.path.getsize(os.path.join(OUTDIR, f)))
             for f in sorted(os.listdir(OUTDIR)) if f.endswith('.npz')]
    for f, s in sizes:
        print(f'  {f:48s} {mb(s):8.1f} MB')
    largest = max((s for _, s in sizes), default=0)

print()
if largest == 0:
    print('Nothing packaged yet.')
elif largest < 50 * 1024**2:
    print(f'Largest file {mb(largest):.1f} MB — under 50 MB.')
    print('COMMIT DIRECTLY to git. No extra hosting needed.')
elif largest < 100 * 1024**2:
    print(f'Largest file {mb(largest):.1f} MB — between 50 and 100 MB.')
    print('Git accepts it but warns, and it stays in history forever.')
    print('Prefer a GitHub RELEASE ASSET.')
else:
    print(f'Largest file {mb(largest):.1f} MB — over the 100 MB git hard limit.')
    print('Use a GitHub RELEASE ASSET (2 GB per file), or Zenodo for a DOI.')

if sizes:
    print('\nRelease asset route: repo -> Releases -> Draft a new release ->')
    print('drag the .npz files in. Then link them from README section 4 and')
    print('uncomment artifacts/predictions/ in .gitignore.')


## 4. Integrity check

In [ ]:
ok, checked = True, 0
if not READY or not inventory:
    print('skipped')
else:
    for m, files in inventory.items():
        out = os.path.join(OUTDIR, f'raw_llm_predictions_{m}_n10000.npz')
        if not os.path.exists(out):
            continue
        z = np.load(out)
        for f in files:
            key = os.path.splitext(os.path.basename(f))[0]
            same = np.array_equal(np.load(f), z[key])
            ok &= same
            checked += 1
            if not same:
                print(f'  MISMATCH {m} {key}')
        print(f'{m:14s} {len(files)} arrays verified')
    print()
    print(f'PASS — {checked} packaged arrays identical to the originals.' if ok
          else 'FAIL — see mismatches above.')
